# Instalando e iniciando pyspark

In [ ]:
# Instalar SDK Java (25 si está disponible, si no 21)
!sudo apt-get update -qq > /dev/null
!sudo apt-get install -y openjdk-25-jdk-headless -qq > /dev/null 2>&1 || \
 sudo apt-get install -y openjdk-21-jdk-headless -qq > /dev/null

# Descargar Spark 4.2.0
!wget -q https://downloads.apache.org/spark/spark-4.2.0/spark-4.2.0-bin-hadoop3.tgz
# Descomprimir el archivo descargado
!tar xf spark-4.2.0-bin-hadoop3.tgz
# Configurar variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-25-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.2.0-bin-hadoop3"
# Desinstalar dataproc-spark-connect
!pip uninstall -y -q dataproc-spark-connect
# Instalar findspark
!pip install -q findspark
# Instalar pyspark
!pip install -q pyspark==4.2.0
# Se importa la libreria findspark
import findspark
findspark.init()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()

# Creando un DataFrame

Un DataFrame es el equivalente a una tabla relacional. Se puede crear de distintas formas usando la sesión de Spark (`SparkSession`).


## Con valores específicos

Para crear un DataFrame con valores específicos, usa el método `createDataFrame`, donde los datos se pasan como una lista de tuplas.

In [ ]:
# Crear DataFrame sin esquema (los tipos se infieren automáticamente)
# data es una lista de tuplas, cada tupla es un renglón
df = spark.createDataFrame(
    data=[("Ana", "Colombia", 34), ("Roberto", "Guatemala", 17)]
)
#df.show()

In [ ]:
df.show()

+-------+---------+---+
|     _1|       _2| _3|
+-------+---------+---+
|    Ana| Colombia| 34|
|Roberto|Guatemala| 17|
+-------+---------+---+



In [ ]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

In [ ]:
# Crear DataFrame especificando nombres de columnas
# schema recibe una lista con los nombres de las columnas
df = spark.createDataFrame(
    data=[("Ana", "Colombia", 34), ("Roberto", "Guatemala", 17)],
    schema=["Nombre", "País", "Edad"]
)

In [ ]:
df.show()

+-------+---------+----+
| Nombre|     País|Edad|
+-------+---------+----+
|    Ana| Colombia|  34|
|Roberto|Guatemala|  17|
+-------+---------+----+



In [ ]:
df.printSchema()

root
 |-- Nombre: string (nullable = true)
 |-- País: string (nullable = true)
 |-- Edad: long (nullable = true)



El tipo de dato fue inferido automáticamente, pero se puede especificar usando un esquema con `StructType` y `StructField`.

Un `StructType()` recibe una lista de elementos StructField. # Poder cambiar esto en algun moemenot
Cada `StructField` define: nombre de columna, tipo de dato y si acepta nulos.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
# StructType() recibe una lista de elementos StructField
# StructField define: nombre de columna, tipo de dato y si acepta nulos


schema = StructType([
    StructField("Nombre", StringType(), True),
    StructField("País", StringType(), True),
    StructField("Edad", IntegerType(), True)
])

df = spark.createDataFrame(
    data=[("Ana", "Colombia", 34), ("Roberto", "Guatemala", None)],
    schema=schema
)
df.show()

+-------+---------+----+
| Nombre|     País|Edad|
+-------+---------+----+
|    Ana| Colombia|  34|
|Roberto|Guatemala|NULL|
+-------+---------+----+



In [ ]:
df.printSchema()

root
 |-- Nombre: string (nullable = true)
 |-- País: string (nullable = true)
 |-- Edad: integer (nullable = true)



In [ ]:
# También se puede crear desde una lista de diccionarios
df = spark.createDataFrame([
    {"Nombre": "Ana", "País": "Colombia", "Edad": 34},
    {"Nombre": "Roberto", "País": "Guatemala", "Edad": 24, "Ocupacion":"Programador"}
])
df.show()

+----+-------+---------+-----------+
|Edad| Nombre|     País|  Ocupacion|
+----+-------+---------+-----------+
|  34|    Ana| Colombia|       NULL|
|  24|Roberto|Guatemala|Programador|
+----+-------+---------+-----------+



## De un archivo CSV

Para ello se utiliza `spark.read.csv("ruta/archivo.csv")`.

In [ ]:
# Lectura básica de un CSV
retail_df = spark.read.csv("/content/Online Retail.csv")
retail_df.show(5)

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|      _c0|      _c1|                 _c2|     _c3|             _c4|      _c5|       _c6|           _c7|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|
|   536365|   85123A|WHITE HANGING HEA...|       6|01/12/2010 08:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|01/12/2010 08:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|01/12/2010 08:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|01/12/2010 08:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
only showing top 5 rows


Nótese que no leyó los nombres de columna correctamente. Para corregirlo se especifica `header=True`.

In [ ]:
# header=True indica que la primera fila contiene los nombres de columnas
retail_df = spark.read.csv("/content/Online Retail.csv", header=True)
retail_df.show(5)
retail_df.printSchema()

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|01/12/2010 08:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|01/12/2010 08:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|01/12/2010 08:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|01/12/2010 08:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|01/12/2010 08:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
only showing top 5 rows
root
 |-- InvoiceNo: string (nu

In [ ]:
type(retail_df)

pyspark.sql.classic.dataframe.DataFrame

Ahora las columnas tienen nombre, pero todos los tipos de dato son `string`. Una forma rápida de corregirlo es con `inferSchema=True`, que deja a Spark detectar los tipos automáticamente.

In [ ]:
# inferSchema=True permite que Spark detecte los tipos de dato automáticamente
df_retail = spark.read.csv(
    "/content/Online Retail.csv",
    header=True,
    inferSchema=True
)
df_retail.show(5)
df_retail.printSchema()

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|01/12/2010 08:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|01/12/2010 08:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|01/12/2010 08:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|01/12/2010 08:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|01/12/2010 08:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
only showing top 5 rows
root
 |-- InvoiceNo: string (nu

Para tener control total sobre los tipos de dato, se puede definir un esquema explícito con `StructType`. Esto es útil cuando `inferSchema` no detecta correctamente algún tipo (por ejemplo, fechas o decimales).

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, DecimalType, StringType, TimestampType

# Cada StructField define: nombre, tipo de dato, si acepta nulos
schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("UnitPrice", DecimalType(10, 2), True),
    StructField("CustomerID", StringType(), True),
    StructField("Country", StringType(), True)
])

# Se pasa el esquema con el parámetro schema
df_retail = spark.read.csv(
    "/content/Online Retail.csv",
    header=True,
    schema=schema,
    timestampFormat="dd/MM/yyyy HH:mm"
)
df_retail.show(5)
df_retail.printSchema()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows
root

**Nota:** También se puede usar la sintaxis alternativa con `.format()`:
```python
spark.read.format("csv").option("header", "true").load("ruta/archivo.csv")
```
Ambas formas son equivalentes.

## De un DataFrame de pandas

In [ ]:
import pandas as pd

# Crear un DataFrame de pandas
pandas_df = pd.DataFrame({
    "Nombre": ["Ana", "Roberto"],
    "País": ["Colombia", "Guatemala"],
    "Edad": [34, 17]
})

# Convertir a DataFrame de Spark
df = spark.createDataFrame(pandas_df)
df.show()

# El esquema se infiere a partir de los tipos de pandas
df.printSchema()

+-------+---------+----+
| Nombre|     País|Edad|
+-------+---------+----+
|    Ana| Colombia|  34|
|Roberto|Guatemala|  17|
+-------+---------+----+

root
 |-- Nombre: string (nullable = true)
 |-- País: string (nullable = true)
 |-- Edad: long (nullable = true)



In [ ]:
type(pandas_df)

pandas.core.frame.DataFrame

In [ ]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

## Ejercicios

1. Crea un DataFrame con las columnas `estado_id`, `estado_nombre`, `poblacion` y `superficie_km2`, con la información de 5 estados de México. Asegúrate de usar el tipo de dato adecuado para cada columna.

2. Descarga la base `movie_dataset.csv` desde Google Classroom y cárgala en Google Colab. Lee el archivo y guarda su contenido en un DataFrame de PySpark. Asegúrate de usar el tipo de dato adecuado para cada columna.


In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

# Solución ejercicio 1

schema = StructType([
    StructField("estado_id", IntegerType(), True),
    StructField("estado_nombre", StringType(), True),
    StructField("poblacion", IntegerType(), True),
    StructField("superficie_km2", IntegerType(), True)
])

df = spark.createDataFrame(
    data=[(1,"Estado de México",9000,9000),
          (2,"Ciudad de México",11000,5000),
          (3,"Chihuahua",5000,300000),
          (4,"Durango",1000,7000),
          (5,"Baja California",7000,8000)
          ],
    schema=schema
)
df.show(5)


+---------+----------------+---------+--------------+
|estado_id|   estado_nombre|poblacion|superficie_km2|
+---------+----------------+---------+--------------+
|        1|Estado de México|     9000|          9000|
|        2|Ciudad de México|    11000|          5000|
|        3|       Chihuahua|     5000|        300000|
|        4|         Durango|     1000|          7000|
|        5| Baja California|     7000|          8000|
+---------+----------------+---------+--------------+



In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, DecimalType, StringType, TimestampType, DoubleType, DateType,LongType

# Solución ejercicio 2

# Cada StructField define: nombre, tipo de dato, si acepta nulos
schema = StructType([
    StructField("index", IntegerType(), True),
    StructField("budget", LongType(), True),
    StructField("genres", StringType(), True),
    StructField("homepage", StringType(), True),
    StructField("id", IntegerType(), True),
    StructField("keywords", StringType(), True),
    StructField("original_language", StringType(), True),
    StructField("original_title", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DecimalType(10, 2), True),
    StructField("release_date", TimestampType(), True),
    StructField("revenue", LongType(), True),
    StructField("runtime", IntegerType(), True),
    StructField("vote_average", DecimalType(10, 2), True),
    StructField("vote_count", IntegerType(), True),
    StructField("director", StringType(), True)
])

# Se pasa el esquema con el parámetro schema
df_retail = spark.read.csv(
    "/content/movie_dataset.csv",
    header=True
    ,
    schema=schema,
    timestampFormat="dd/MM/yyyy"
)
df_retail.show(20)
df_retail.printSchema()


+-----+---------+--------------------+--------------------+------+--------------------+-----------------+--------------------+--------------------+----------+-------------------+----------+-------+------------+----------+-----------------+
|index|   budget|              genres|            homepage|    id|            keywords|original_language|      original_title|            overview|popularity|       release_date|   revenue|runtime|vote_average|vote_count|         director|
+-----+---------+--------------------+--------------------+------+--------------------+-----------------+--------------------+--------------------+----------+-------------------+----------+-------+------------+----------+-----------------+
|    0|237000000|Action Adventure ...|http://www.avatar...| 19995|culture clash fut...|               en|              Avatar|In the 22nd centu...|    150.44|2009-12-10 00:00:00|2787965087|    162|        7.20|     11800|    James Cameron|
|    1|300000000|Adventure Fantasy...|ht

# Explorando un DataFrame

Antes de trabajar con un DataFrame, es útil conocer su estructura y contenido. PySpark ofrece varios métodos para inspeccionar los datos rápidamente.

In [ ]:
# Mostrar las primeras filas
df_retail.show(5)

+-----+---------+--------------------+--------------------+------+--------------------+-----------------+--------------------+--------------------+----------+-------------------+----------+-------+------------+----------+-----------------+
|index|   budget|              genres|            homepage|    id|            keywords|original_language|      original_title|            overview|popularity|       release_date|   revenue|runtime|vote_average|vote_count|         director|
+-----+---------+--------------------+--------------------+------+--------------------+-----------------+--------------------+--------------------+----------+-------------------+----------+-------+------------+----------+-----------------+
|    0|237000000|Action Adventure ...|http://www.avatar...| 19995|culture clash fut...|               en|              Avatar|In the 22nd centu...|    150.44|2009-12-10 00:00:00|2787965087|    162|        7.20|     11800|    James Cameron|
|    1|300000000|Adventure Fantasy...|ht

In [ ]:
# Ver el esquema (nombre y tipo de cada columna)
df_retail.printSchema()

root
 |-- index: integer (nullable = true)
 |-- budget: long (nullable = true)
 |-- genres: string (nullable = true)
 |-- homepage: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- keywords: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: decimal(10,2) (nullable = true)
 |-- release_date: timestamp (nullable = true)
 |-- revenue: long (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- vote_average: decimal(10,2) (nullable = true)
 |-- vote_count: integer (nullable = true)
 |-- director: string (nullable = true)



In [ ]:
# Obtener la lista de nombres de columnas
df_retail.columns

['index',
 'budget',
 'genres',
 'homepage',
 'id',
 'keywords',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'release_date',
 'revenue',
 'runtime',
 'vote_average',
 'vote_count',
 'director']

In [ ]:
# Obtener nombre y tipo de cada columna como lista de tuplas
df_retail.dtypes

[('index', 'int'),
 ('budget', 'bigint'),
 ('genres', 'string'),
 ('homepage', 'string'),
 ('id', 'int'),
 ('keywords', 'string'),
 ('original_language', 'string'),
 ('original_title', 'string'),
 ('overview', 'string'),
 ('popularity', 'decimal(10,2)'),
 ('release_date', 'timestamp'),
 ('revenue', 'bigint'),
 ('runtime', 'int'),
 ('vote_average', 'decimal(10,2)'),
 ('vote_count', 'int'),
 ('director', 'string')]

In [ ]:
# Contar el número total de renglones
df_retail.count()

541909

In [ ]:
# Resumen estadístico de las columnas numéricas (count, mean, stddev, min, max)
df_retail.describe().show()

+-------+------------------+------------------+--------------------+-----------------+-----------------+------------------+-----------+
|summary|         InvoiceNo|         StockCode|         Description|         Quantity|        UnitPrice|        CustomerID|    Country|
+-------+------------------+------------------+--------------------+-----------------+-----------------+------------------+-----------+
|  count|            541909|            541909|              540455|           541909|           541909|            406829|     541909|
|   mean|  559965.752026781|27623.240210938104|             20713.0| 9.55224954743324|         4.611114|15287.690570239585|       NULL|
| stddev|13428.417280799484| 16799.73762842768|                NULL|218.0811578502348|96.75985306153139|1713.6003033215932|       NULL|
|    min|            536365|             10002| 4 PURPLE FLOCK D...|           -80995|        -11062.06|             12346|  Australia|
|    max|           C581569|                 m| 

In [ ]:
# Resumen estadístico de columnas específicas
df_retail.describe("Quantity", "UnitPrice").show()

+-------+-----------------+-----------------+
|summary|         Quantity|        UnitPrice|
+-------+-----------------+-----------------+
|  count|           541909|           541909|
|   mean| 9.55224954743324|         4.611114|
| stddev|218.0811578502348|96.75985306153139|
|    min|           -80995|        -11062.06|
|    max|            80995|         38970.00|
+-------+-----------------+-----------------+



**Tip:** Acostúmbrate al inicio usar `printSchema()` y `show()` cada vez que creas o transformas un DataFrame. Es la forma más rápida de verificar que los datos se ven como esperas.

# Operaciones con columnas

## Seleccionar columnas

En PySpark se seleccionan columnas con el método `.select()`.

In [68]:
# Recordatorio de los datos
df_retail.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


In [69]:
# Seleccionar una columna por nombre
df_retail.select("Description").show(5)

+--------------------+
|         Description|
+--------------------+
|WHITE HANGING HEA...|
| WHITE METAL LANTERN|
|CREAM CUPID HEART...|
|KNITTED UNION FLA...|
|RED WOOLLY HOTTIE...|
+--------------------+
only showing top 5 rows


In [70]:
# Seleccionar varias columnas
df_retail.select("Description", "Quantity", "InvoiceDate").show(5)

+--------------------+--------+-------------------+
|         Description|Quantity|        InvoiceDate|
+--------------------+--------+-------------------+
|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|
| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|
|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|
|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|
|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|
+--------------------+--------+-------------------+
only showing top 5 rows


In [71]:
# También se puede usar col() para referenciar columnas
from pyspark.sql.functions import col

df_retail.select(col("Description"), col("Quantity")).show(5)

+--------------------+--------+
|         Description|Quantity|
+--------------------+--------+
|WHITE HANGING HEA...|       6|
| WHITE METAL LANTERN|       6|
|CREAM CUPID HEART...|       8|
|KNITTED UNION FLA...|       6|
|RED WOOLLY HOTTIE...|       6|
+--------------------+--------+
only showing top 5 rows


In [72]:
# También se puede usando la notación de un atributo del dataframe
df_retail.select(df_retail.Description, df_retail.Quantity).show()

+--------------------+--------+
|         Description|Quantity|
+--------------------+--------+
|WHITE HANGING HEA...|       6|
| WHITE METAL LANTERN|       6|
|CREAM CUPID HEART...|       8|
|KNITTED UNION FLA...|       6|
|RED WOOLLY HOTTIE...|       6|
|SET 7 BABUSHKA NE...|       2|
|GLASS STAR FROSTE...|       6|
|HAND WARMER UNION...|       6|
|HAND WARMER RED P...|       6|
|ASSORTED COLOUR B...|      32|
|POPPY'S PLAYHOUSE...|       6|
|POPPY'S PLAYHOUSE...|       6|
|FELTCRAFT PRINCES...|       8|
|IVORY KNITTED MUG...|       6|
|BOX OF 6 ASSORTED...|       6|
|BOX OF VINTAGE JI...|       3|
|BOX OF VINTAGE AL...|       2|
|HOME BUILDING BLO...|       3|
|LOVE BUILDING BLO...|       3|
|RECIPE BOX WITH M...|       4|
+--------------------+--------+
only showing top 20 rows


In [73]:
# Seleccionar utilizando notación de index en el dataframe
df_retail.select(df_retail['Description'], df_retail['Quantity']).show()

+--------------------+--------+
|         Description|Quantity|
+--------------------+--------+
|WHITE HANGING HEA...|       6|
| WHITE METAL LANTERN|       6|
|CREAM CUPID HEART...|       8|
|KNITTED UNION FLA...|       6|
|RED WOOLLY HOTTIE...|       6|
|SET 7 BABUSHKA NE...|       2|
|GLASS STAR FROSTE...|       6|
|HAND WARMER UNION...|       6|
|HAND WARMER RED P...|       6|
|ASSORTED COLOUR B...|      32|
|POPPY'S PLAYHOUSE...|       6|
|POPPY'S PLAYHOUSE...|       6|
|FELTCRAFT PRINCES...|       8|
|IVORY KNITTED MUG...|       6|
|BOX OF 6 ASSORTED...|       6|
|BOX OF VINTAGE JI...|       3|
|BOX OF VINTAGE AL...|       2|
|HOME BUILDING BLO...|       3|
|LOVE BUILDING BLO...|       3|
|RECIPE BOX WITH M...|       4|
+--------------------+--------+
only showing top 20 rows


In [74]:
# Seleccionar todas las columnas
df_retail.select('*').show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|2010-12-01 08:26:00|     7.65|     17850|United Kingdom|
|   536365|    21730|GLASS S

**Nota:** Existen varias formas de referenciar columnas, como `df.columna` o `df["columna"]`, pero usar strings o `col()` son las más comunes.

## Crear una columna

En PySpark se crea una nueva columna con el método `.withColumn()`. El DataFrame original no se modifica, se retorna uno nuevo.

In [75]:
# Crear una columna con una operación aritmética sobre otras columnas
# Recuerda: col() se importó antes con from pyspark.sql.functions import col
df_cost = df_retail.withColumn("TotalCost", col("Quantity") * col("UnitPrice"))
df_cost.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalCost|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|    15.30|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|    20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|    22.00|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|    20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|    20.34|
+---------+---------+--------------------+--------+-----

In [77]:
# Crear una columna booleana a partir de una condición
df_quantity = df_retail.withColumn("HighQuantity", col("Quantity") > 7)
df_quantity.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|HighQuantity|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|       false|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|       false|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|        true|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|       false|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|       false|
+---------+---------+-----------

In [ ]:
# El DataFrame original no se modificó (los DataFrames en PySpark son inmutables)
df_retail.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


## Castear una columna

Para cambiar el tipo de dato de una columna se usa el método `.cast()` sobre la columna.

In [ ]:
# Ver el tipo actual de las columnas
df_retail.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



In [ ]:
from pyspark.sql.types import DoubleType

# Cambiar UnitPrice de Decimal a Double
df_casted = df_retail.withColumn("UnitPrice", col("UnitPrice").cast(DoubleType()))
df_casted.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



In [ ]:
# También se puede castear usando el nombre del tipo como string
df_casted = df_retail.withColumn("CustomerID", col("CustomerID").cast("integer"))
df_casted.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



## Renombrar una columna

Para renombrar una columna se usa `.withColumnRenamed("nombre_actual", "nombre_nuevo")`.

In [ ]:
# Recordatorio: df_cost tiene la columna TotalCost que creamos antes
df_cost.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalCost|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|    15.30|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|    20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|    22.00|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|    20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|    20.34|
+---------+---------+--------------------+--------+-----

In [ ]:
# Renombrar TotalCost a Total
df_renamed = df_cost.withColumnRenamed("TotalCost", "Total")
df_renamed.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|Total|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|15.30|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|22.00|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|20.34|
+---------+---------+--------------------+--------+-------------------+---------+-------

In [ ]:
# Si la columna no existe, PySpark no lanza error — simplemente no hace nada
df_cost.withColumnRenamed("ColumnaQueNoExiste", "Otra").show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalCost|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|    15.30|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|    20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|    22.00|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|    20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|    20.34|
+---------+---------+--------------------+--------+-----

## Eliminar columnas

Para eliminar una o más columnas se usa el método `.drop()`.

In [ ]:
# Eliminar una columna
df_renamed.drop("Total").show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


In [ ]:
# Recuerda: el DataFrame original no se modifica
df_renamed.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|Total|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|15.30|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|22.00|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|20.34|
+---------+---------+--------------------+--------+-------------------+---------+-------

In [ ]:
# Para conservar el cambio, hay que asignarlo a una variable
df_clean = df_renamed.drop("Total")
df_clean.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


In [ ]:
# Eliminar varias columnas a la vez
df_clean = df_renamed.drop("InvoiceNo", "StockCode", "Description")
df_clean.show(5)

+--------+-------------------+---------+----------+--------------+-----+
|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|Total|
+--------+-------------------+---------+----------+--------------+-----+
|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|15.30|
|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|20.34|
|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|22.00|
|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|20.34|
|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|20.34|
+--------+-------------------+---------+----------+--------------+-----+
only showing top 5 rows


# Operaciones con renglones

## Filtrar renglones

Para filtrar renglones se usa el método `.filter()`. Las condiciones se construyen sobre columnas.

In [ ]:
# Recordatorio de los datos
df_retail.show(5)

In [ ]:
# Filtrar renglones donde Quantity es mayor a 10
df_retail.filter(col("Quantity") > 10).show(5)

In [ ]:
# Filtrar con igualdad
df_retail.filter(col("Quantity") == 10).show(5)

In [ ]:
# Combinar condiciones con AND (operador &)
# Cada condición debe ir entre paréntesis
df_retail.filter(
    (col("Quantity") > 10) & (col("Country") == "United Kingdom")
).show(5)

In [ ]:
# Combinar condiciones con OR (operador |)
df_retail.filter(
    (col("Quantity") > 10) | (col("Country") == "United Kingdom")
).show(5)

In [ ]:
# Negar una condición con ~ (NOT)
df_retail.filter(
    ~(col("Quantity") > 10)
).show(5)

**Nota:** `.where()` es un alias de `.filter()`, ambos hacen lo mismo.

## Ordenar renglones

Para ordenar renglones se usa `.orderBy()`. Por defecto ordena de forma ascendente.

**Nota:** `.sort()` es un alias de `.orderBy()`, ambos hacen lo mismo.

In [ ]:
# Ordenar de forma ascendente por Quantity
df_retail.orderBy("Quantity").show(5)

In [ ]:
# Ordenar de forma descendente con .desc()
df_retail.orderBy(col("Quantity").desc()).show(5)

In [ ]:
# Ordenar por múltiples columnas
df_retail.orderBy(col("Quantity").desc(), col("InvoiceDate").desc()).show(5)

## Remover renglones duplicados

Para eliminar renglones duplicados se usa `.dropDuplicates()`. Se puede aplicar sobre todas las columnas o solo sobre algunas.

In [ ]:
# Total de renglones en el DataFrame original
df_retail.count()

In [ ]:
# Remover duplicados considerando todas las columnas
df_retail.dropDuplicates().count()

In [ ]:
# Remover duplicados considerando solo ciertas columnas
df_retail.dropDuplicates(["InvoiceNo", "StockCode"]).count()

In [ ]:
df_pair_unicos = df_retail.dropDuplicates(["InvoiceNo", "StockCode"])

In [ ]:
df_pair_unicos.show(5)

**Nota:** `.distinct()` es equivalente a `.dropDuplicates()` sin parámetros (aplica sobre todas las columnas). La ventaja de `dropDuplicates()` es que permite especificar columnas.

## Manejar valores nulos

Para eliminar filas con valores nulos se usa `na.drop()`. Se puede especificar si se eliminan filas con *algún* valor nulo (`"any"`) o solo donde *todos* sean nulos (`"all"`).

In [ ]:
# Total de renglones en el DataFrame original
df_retail.count()

In [ ]:
# Eliminar renglones que tengan algún valor nulo (comportamiento por defecto)
df_retail.na.drop("any").count()

In [ ]:
# Verificación: filtrar renglones donde algún valor sea nulo
df_any_nulls = df_retail.filter(
    col("InvoiceNo").isNull() |
    col("StockCode").isNull() |
    col("Description").isNull() |
    col("Quantity").isNull() |
    col("InvoiceDate").isNull() |
    col("UnitPrice").isNull() |
    col("CustomerID").isNull() |
    col("Country").isNull()
)
df_any_nulls.count()

In [ ]:
df_retail.show(3)

In [ ]:
# Eliminar renglones donde todos los valores sean nulos
df_retail.na.drop("all").count()

In [ ]:
# Verificación: filtrar renglones donde todos los valores sean nulos
df_all_nulls = df_retail.filter(
    col("InvoiceNo").isNull() &
    col("StockCode").isNull() &
    col("Description").isNull() &
    col("Quantity").isNull() &
    col("InvoiceDate").isNull() &
    col("UnitPrice").isNull() &
    col("CustomerID").isNull() &
    col("Country").isNull()
)
df_all_nulls.count()

In [ ]:
# Eliminar renglones con nulos solo en columnas específicas
df_retail.na.drop(subset=["CustomerID", "Description"]).count()

## Agregar renglones

Para agregar renglones a un DataFrame se usa `.union()`. Ambos DataFrames deben tener el mismo esquema.

In [ ]:
from datetime import datetime

# Crear un DataFrame nuevo con tipos compatibles
new_df = spark.createDataFrame(
    data=[("INV12345", "A123", "Product A", 5, datetime(2022, 1, 1, 12, 0), 10.0, "12345", "UK")],
    schema=["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
)

# Unir ambos DataFrames
combined_df = df_retail.union(new_df)
print("Renglones originales:", df_retail.count(), "| Después de union:", combined_df.count())

In [ ]:
# Verificar que el nuevo renglón se agregó
combined_df.filter(col("InvoiceNo") == "INV12345").show()

### Cuidado con el orden de las columnas

`.union()` une por **posición**, no por nombre de columna. Si el orden es diferente, los datos se mezclan sin dar error.

In [ ]:
# Crear un DataFrame con columnas en diferente orden
new_df = spark.createDataFrame(
    data=[("INV12345", "A123", "Product A", 5, datetime(2022, 1, 1, 12, 0), "UK", "12345", 10.0)],
    schema=["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "Country", "CustomerID", "UnitPrice"]
)

#combined_df = df_retail.union(new_df)
#combined_df.filter(col("InvoiceNo") == "INV12345").show()

In [ ]:
# unionByName() resuelve esto — une por nombre de columna, no por posición
combined_df = df_retail.unionByName(new_df)
combined_df.filter(col("InvoiceNo") == "INV12345").show()

## Ejercicios

1. De la base `movie_dataset.csv`, filtra las películas con un presupuesto (`budget`) mayor a 100 millones y selecciona las columnas `original_title`, `budget`, `revenue`, `director` y `release_date`.

2. Usando el DataFrame del ejercicio anterior, crea una nueva columna llamada `profit` calculada como `revenue - budget`. Además, crea una columna `year` que contenga el año extraído de `release_date`. Muestra las columnas `original_title`, `budget`, `revenue`, `profit` y `year`.

3. Filtra las películas con `profit` mayor a 200 millones y ordénalas por `year` de forma ascendente.

In [ ]:
from pyspark.sql.functions import col

# Solución ejercicio 1


In [ ]:
from pyspark.sql.functions import col, year

# Solución ejercicio 2


In [ ]:
# Solución ejercicio 3


# Joins

Para unir dos DataFrames se usa el método `.join()`. Se especifica el tipo de unión con `how` y las columnas de unión con `on`.

Tipos de unión comunes:
- `inner`: devuelve solo las filas con coincidencia en ambos DataFrames (es el tipo por defecto).
- `left`: mantiene todas las filas del primer DataFrame y solo las coincidencias del segundo.
- `right`: mantiene todas las filas del segundo DataFrame y solo las coincidencias del primero.
- `full`: mantiene todas las filas de ambos DataFrames, rellenando con nulos donde no hay coincidencia.

In [ ]:
# Crear dos DataFrames para ilustrar los joins
df1 = spark.createDataFrame(
    [(1, "Alice"), (2, "Bob"), (3, "Charlie")],
    schema=["id", "name"]
)

df2 = spark.createDataFrame(
    [(2, "HR"), (3, "Finance"), (4, "IT")],
    schema=["id", "department"]
)

df1.show()
df2.show()

In [ ]:
# inner join: solo filas con id en ambos DataFrames
df1.join(df2, on="id", how="inner").show()

In [ ]:
# left join: todas las filas de df1, coincidencias de df2
df1.join(df2, on="id", how="left").show()

In [ ]:
# right join: todas las filas de df2, coincidencias de df1
df1.join(df2, on="id", how="right").show()

In [ ]:
# full join: todas las filas de ambos, nulos donde no hay coincidencia
df1.join(df2, on="id", how="full").show()

### Joins con columnas de diferente nombre

Si las columnas de unión tienen diferente nombre en cada DataFrame, se especifica la condición directamente:
```python
df_joined = df_B.join(df_A, on=df_A["col1"] == df_B["col2"], how="inner")
```

Si el join involucra más de una columna:
```python
df_joined = df_B.join(
    df_A,
    on=(df_A["col1"] == df_B["col2"]) & (df_A["col3"] == df_B["col4"]),
    how="inner"
)
```

# Agregaciones

Para agregar datos se usa `.groupBy()` (similar a `GROUP BY` en SQL) combinado con `.agg()`. Las funciones de agregación se importan desde `pyspark.sql.functions`.

In [ ]:
# Crear un DataFrame de ejemplo
df_emp = spark.createDataFrame(
    [("Alice", "Ventas", 5000, 5),
     ("Bob", "IT", 4000, 3),
     ("Charlie", "Ventas", 6000, 8),
     ("David", "IT", 3500, 3),
     ("Eva", "Ventas", 7000, 10)],
    schema=["name", "department", "salary", "experience"]
)
df_emp.show()

In [ ]:
from pyspark.sql.functions import avg, sum, max, min, count

# Agregación sin groupBy — se aplica a todo el DataFrame
df_emp.agg(avg("salary").alias("avg_salary")).show()

In [ ]:
# Agrupar por departamento y calcular el promedio salarial
df_emp.groupBy("department").agg(
    avg("salary").alias("avg_salary")
).show()

In [ ]:
# Múltiples agregaciones sobre el mismo grupo
df_emp.groupBy("department").agg(
    avg("salary").alias("avg_salary"),
    sum("salary").alias("total_salary"),
    max("salary").alias("max_salary"),
    count("name").alias("num_employees")
).show()

**Nota:** La lista completa de funciones de agregación está en la [documentación oficial](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#aggregate-functions).

## Ejercicios

1. De `df_movies`, elimina los renglones donde `director` sea nulo o `"0"`. Agrupa por `director`, cuenta cuántas películas dirigió cada uno, ordena de forma descendente y muestra los 5 directores con más películas.

2. Con los 5 directores del ejercicio anterior, crea un nuevo DataFrame con las columnas `director` y `nationality`. Investiga la nacionalidad de cada director.

3. Realiza un join entre `df_movies` y el DataFrame de directores. Muestra `original_title`, `director` y `nationality`.

In [ ]:
from pyspark.sql.functions import col, count

# Solución ejercicio 1


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

# Solución ejercicio 2


In [ ]:
# Solución ejercicio 3


**Ojo**: cuidado con pyspark `string` != `int`
"In PySpark, comparing a string column with an integer using the != operator might lead to unexpected behavior due to implicit type casting. When you compare a string column with an integer, Spark attempts to cast the string column to an integer for comparison. This can lead to unexpected results, especially if the string column contains non-numeric values."

# Llamadas encadenadas (chaining)

En PySpark, los métodos que transforman un DataFrame devuelven un nuevo DataFrame. Esto permite encadenar varias operaciones en una sola expresión, lo cual mejora la legibilidad.

**Ejemplo 1:** ¿Cuál es la cantidad total de artículos y el precio promedio por país, considerando solo cantidades mayores a cero?

In [ ]:
from pyspark.sql.functions import col, sum, avg, count

In [ ]:
# Paso a paso: filter -> groupBy -> agg -> sort
result_df = (df_retail
    .filter(col("Quantity") > 0)
    .groupBy("Country")
    .agg(
        sum("Quantity").alias("total_quantity"),
        avg("UnitPrice").alias("avg_price")
    )
    .sort("total_quantity", ascending=False)
)

result_df.show(10)

**Ejemplo 2:** Para clientes del Reino Unido, ¿cuánto gastó cada uno y cuántas transacciones realizó?

In [ ]:
# filter → withColumn → groupBy → agg → sort
result_uk = (df_retail
    .filter(
        (col("Country") == "United Kingdom") & col("CustomerID").isNotNull()
    )
    .withColumn("TotalPrice", col("Quantity") * col("UnitPrice"))
    .groupBy("CustomerID")
    .agg(
        sum("TotalPrice").alias("total_spent"),
        count("InvoiceNo").alias("num_transactions")
    )
    .sort("total_spent", ascending=False)
)

result_uk.show(10)

**Tip:** Conviene filtrar nulos y datos inválidos al inicio de la cadena, antes de agrupar. Así se evitan problemas y se procesan menos datos.

## Ejercicios

1. Filtra las películas que contengan el género `"Action"` en la columna `genres`. Calcula la popularidad promedio y el número de películas. Muestra el resultado en un solo DataFrame.

2. Calcula el ingreso total (`revenue`) y el número de películas por `director`. Muestra solo los directores con más de 5 películas, ordenados por ingreso total descendente.

3. Calcula el ingreso total y la popularidad promedio de las películas por año (extraído de `release_date`). Filtra solo los años con más de 10 películas y ordena por ingreso total descendente.

In [ ]:
from pyspark.sql.functions import col, avg, count

# Solución ejercicio 1


In [ ]:
from pyspark.sql.functions import col, sum, count

# Solución ejercicio 2


In [ ]:
from pyspark.sql.functions import col, sum, avg, count, year

# Solución ejercicio 3
